In [1]:
!pip uninstall -y torchao
!pip install torchao>=0.16.0

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 9.4 MB/s  0:00:00


In [4]:
!pip install -U torchao

In [5]:
# !pip uninstall -y torchao
# !pip install -q "torchao>=0.16.0"
# !pip install -q -U "bitsandbytes>=0.46.1" trl peft

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
SYSTEM = "You are VoltStore's support assistant. Reply in the VoltStore format."

def chat(prompt, history=None):
    messages = [{"role": "system", "content": SYSTEM}] + (history or []) + [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.8
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    print(answer)
    return answer

In [8]:
# BEFORE TRAINING
chat('my usb hub arrived with a cracked case')
print('-' * 70)
chat("I want to send back my headphones, I got them 3 weeks ago.")
print('-' * 70)
chat("I returned my webcam 5 days ago and still no refund")
print('-' * 70)
chat("My smart plug keeps disconnecting from wifi.")

I'm sorry to hear that you encountered an issue with your USB hub, especially when it comes to its casing. This is unfortunate but not uncommon for electronics products. We recommend reaching out to the seller or manufacturer directly for assistance. They may be able to provide a replacement product or offer compensation based on their policies. If this isn't possible, consider purchasing another compatible USB hub from a trusted retailer who offers easy returns or exchanges if necessary. Let us know how we can assist further in any way!
----------------------------------------------------------------------
Hello! Thank you for reaching out to us at VoltStore. We're here to assist with any issues or concerns you might have regarding your purchase.

If you've received the headphones and they haven't arrived as expected within a few days after shipping, it would be best to check our order status through our website or app. If you've already contacted our customer service team about this 

'It sounds like your smart plug is experiencing connectivity issues with the Wi-Fi network it is connected to. Here are a few steps you can try to resolve this issue:\n\n1. **Check for Updates**: Ensure that both your router and smart plug are up-to-date with their firmware.\n\n2. **Reset Network Settings**:\n   - On your device (smart plug), go into the settings menu.\n   - Look for an option labeled "Network" or "Wi-Fi".\n   - Reset the network settings by selecting "Factory Reset".\n\n3. **Reconnect Devices**: Try reconnecting all devices in your home to the same Wi-Fi network, including your router, other smart plugs, and any other devices.\n\n4. **Restart Your Router and Smart Plug**: Sometimes, simply restarting these devices can help fix temporary connectivity issues.\n\n5. **Try Different Networks**: If possible, switch between different networks on your router to see if the problem persists on another network.\n\n6. **Check Device Logs**: Some smart plugs have logs where you c

In [10]:
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import torch

ds = load_dataset("json", data_files={
    "train": "voltstore_train.jsonl",
    "validation": "voltstore_val_heldout_topics.jsonl",
})

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

cfg = SFTConfig(
    output_dir="qwen-voltstore-v2",

    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.03,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    bf16=False,
    max_length=1024,
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=cfg,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    peft_config=LoraConfig(
        r=16,
        lora_alpha=64,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
    ),
)
trainer.train()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.605486,1.068386,0.959344,0.746420,27087.000000
2,0.712949,1.064311,0.666962,0.770020,54174.000000
3,0.519889,1.053613,0.642768,0.774022,81261.000000


TrainOutput(global_step=33, training_loss=0.8940561937563347, metrics={'train_runtime': 213.2027, 'train_samples_per_second': 2.28, 'train_steps_per_second': 0.155, 'total_flos': 735366210299904.0, 'train_loss': 0.8940561937563347, 'epoch': 3.0})

In [11]:
# AFTER TRAINING
chat('my usb hub arrived with a cracked case')
print('-' * 70)
chat("I want to send back my headphones, I got them 3 weeks ago.")
print('-' * 70)
chat("I returned my webcam 5 days ago and still no refund")
print('-' * 70)
chat("My smart plug keeps disconnecting from wifi.")

Hi there,

Thanks for reaching out about your USB hub arriving damaged.

Here's what to do:
1. Photograph the damage to the product, the shipping label and the outer packaging.
2. Send the photos in a reply, along with your order number, within 14 days of delivery.
3. Tell us whether you'd like a free replacement or a full refund.

Damage in transit is always on us, and return shipping is free.

Best regards,
VoltStore Support
----------------------------------------------------------------------
Hi there,

Thanks for reaching out about returning your wireless headphones.

Here's what to do:
1. Log in and open My Orders, select the order and choose Request Return.
2. Print the prepaid return label and pack the wireless headphones with everything that came in the box.
3. Drop the parcel off with the carrier. Your refund is issued 5–7 business days after we receive it.

Return shipping is free, but you pay for returned items.

Best regards,
VoltStore Support
-----------------------------

"Hi there,\n\nThanks for reaching out about your smart plug losing connection.\n\nHere's what to do:\n1. Wait 30 seconds, then check that it turns on and off with a switch.\n2. Restart both your phone and the smart plug.\n3. Change the Wi-Fi channel or try using a different network.\n\nIf the problem persists after that, reply with your order number so we can confirm your purchase date.\n\nBest regards,\nVoltStore Support"

In [12]:
trainer.save_model("adapter")
tokenizer.save_pretrained("adapter")

('adapter/tokenizer_config.json',
 'adapter/chat_template.jinja',
 'adapter/tokenizer.json')

In [13]:
!zip -r adapter.zip adapter
from google.colab import files
files.download("adapter.zip")

  adding: adapter/ (stored 0%)
  adding: adapter/tokenizer.json (deflated 81%)
  adding: adapter/tokenizer_config.json (deflated 59%)
  adding: adapter/training_args.bin (deflated 53%)
  adding: adapter/README.md (deflated 65%)
  adding: adapter/chat_template.jinja (deflated 71%)
  adding: adapter/adapter_model.safetensors (deflated 54%)
  adding: adapter/adapter_config.json (deflated 61%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>